# Model training and evaluation

This notebook demonstrates two approaches: **supervised learning**, where linear regression predicts a known target (a student's final grade), and **unsupervised learning**, where K-means groups similar flowers without using their species labels. Run the cells from top to bottom.

## 1. Import libraries

Pandas handles tabular data, NumPy supports numerical calculations, Matplotlib creates plots, and scikit-learn provides preprocessing, models, and evaluation metrics.

In [ ]:
# ===========================
# Cell 1: Import libraries
# ===========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


## Student performance dataset

`student-mat.csv` is the mathematics dataset from the [UCI Student Performance dataset](https://archive.ics.uci.edu/dataset/320/student+performance). The local file contains 395 student records and 33 columns, covering grades, study habits, and personal background. Each row represents one student.

In this notebook, we use five input features to predict the final mathematics grade (`G3`) with linear regression:

| Column | Meaning |
| --- | --- |
| `G1` | First-period grade (0-20). |
| `G2` | Second-period grade (0-20). |
| `studytime` | Weekly study time category: 1 = under 2 hours, 2 = 2-5 hours, 3 = 5-10 hours, 4 = over 10 hours. |
| `failures` | Number of past class failures. |
| `absences` | Number of school absences. |
| `G3` | Final grade (0-20), the prediction target. |

The CSV uses semicolons as separators, so we load it with `sep=';'`.


In [2]:
import pandas as pd

# Load student-mat.csv
data = pd.read_csv("../datasets/student-mat.csv", sep=';')

# Select relevant columns
data = data[['G1','G2','G3','studytime','failures','absences']]
data.head()


,G1,G2,G3,studytime,failures,absences
0,5,6,6,2,0,6
1,5,5,6,2,0,4
2,7,8,10,2,3,10
3,15,14,15,3,0,2
4,6,10,10,2,0,4


## 2. Prepare the training and test data

`X` contains the five input features; `y` contains the final grade to predict. We use 80% of the records for training and reserve 20% for testing on unseen students. `random_state=42` makes this split reproducible.

`StandardScaler` centers each feature around zero and scales it using its training-set standard deviation. Fit the scaler only on the training data, then reuse it for the test data to avoid leaking information from the test set. Scaling is not required for ordinary linear regression, but demonstrates a reusable preprocessing workflow.

Because `G1` and `G2` are inputs, this model can only be used once those grades are available.

In [ ]:
# ===========================
# Cell 3: Split features & target
# ===========================
X = data[['G1','G2','studytime','failures','absences']]
y = data['G3']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ===========================
# Cell 4: Scale features
# ===========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Explore the relationship between grades

Each point represents a training student: the horizontal axis shows `G1`, and the vertical axis shows `G3`. Look for an upward trend, unusual points, and the spread of final grades at each first-period grade. This plot shows one relationship; the model will use all five features.

In [ ]:
# ===========================
# Cell 5: Visualize relation G1 vs G3
# ===========================
plt.scatter(X_train['G1'], y_train, color='blue', alpha=0.5, label='Train')
# plt.scatter(X_test['G1'], y_test, color='red', alpha=0.5, label='Test')
plt.xlabel("G1")
plt.ylabel("G3 (Final Grade)")
plt.title("G1 vs G3: Linear relation?")
plt.legend()
plt.show()


## 4. Train a linear regression model

The model predicts the final grade as a weighted sum of the scaled features plus an intercept. Calling `fit` learns these weights from the training data by minimizing squared prediction errors. A relationship learned from these observations does not establish cause and effect.

In [ ]:

# ===========================
# Cell 6: Train Linear Regression
# ===========================
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)


## 5. Predict grades for the test set

`predict` applies the trained model to the scaled test features. The resulting `y_pred` values can be compared with the held-out grades in `y_test`; these grades were not used to fit the model.

In [ ]:
# ===========================
# Cell 7: Predictions on test set
# ===========================
y_pred = lr_model.predict(X_test_scaled)



### Compare actual and predicted grades

Red points show the actual final grades and green points show predictions, both plotted against `G1`. Smaller vertical differences for the same student indicate better predictions, although overlapping points can make individual comparisons difficult. This is a prediction comparison, not a training convergence plot; predictions also depend on the other four features.

In [ ]:
# ===========================
# Cell 8: Plot convergence / fit
# ===========================
plt.scatter(X_test['G1'], y_test, color='red', label='Actual')
plt.scatter(X_test['G1'], y_pred, color='green', alpha=0.5, label='Predicted')
plt.xlabel("G1")
plt.ylabel("G3")
plt.title("Linear Regression Predictions vs Actual")
plt.legend()
plt.show()

## 6. Evaluate regression performance

- **MSE:** average squared prediction error. Lower is better; large errors receive extra weight.
- **RMSE:** square root of MSE, expressed in grade points. Lower is better and zero means perfect predictions.
- **R-squared (`R2`):** compares squared errors with predicting the mean of the test targets. A score of 1 is perfect, 0 matches that baseline, and a negative score is worse.

Interpret these metrics together with the plot. They describe performance on this particular test split, so results may differ for another split or student population.

In [ ]:

# ===========================
# Cell 9: Evaluate model
# ===========================
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.2f}")


## 7. Predict a new student's grade

The example student has grades `G1=12` and `G2=13`, study-time category `2`, no past class failures, and three absences. Keep the same feature names and order, and apply the existing scaler before predicting. Do not fit a new scaler to this single student.

The output is a continuous estimate, not a guaranteed grade. Linear regression does not automatically restrict predictions to the valid 0-20 range.

In [ ]:
# ===========================
# Cell 10: Predict new student
# ===========================
new_student = pd.DataFrame([[12, 13, 2, 0, 3]],
                           columns=['G1', 'G2', 'studytime', 'failures', 'absences'])
new_student_scaled = scaler.transform(new_student)
predicted_grade = lr_model.predict(new_student_scaled)
print(f"Predicted final grade (G3) for new student: {predicted_grade[0]:.2f}")

## 8. K-means clustering

We now switch from predicting grades to grouping similar flowers. K-means is an unsupervised method: it learns groups from feature values without using target labels. The following cells import the clustering tools and load the Iris dataset.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris


### Load the Iris data

The input features describe sepal length, sepal width, petal length, and petal width. `iris.target` contains the known species labels for reference, but these are not passed to K-means. `df.head()` provides a first look at the measurements.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target  # for reference only, not used in clustering
feature_names = iris.feature_names

# Optional: convert to DataFrame for easy visualization
df = pd.DataFrame(X, columns=feature_names)
df.head()


### Scale the features

K-means uses distances between samples, so feature scales affect the result. Standardization gives each feature a comparable scale. Here we cluster the full dataset for exploration, so the scaler is fitted to all samples.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


### Fit K-means with three clusters

K-means alternates between assigning samples to the nearest centroid and updating each centroid to the mean of its assigned samples. We choose `n_clusters=3` for this demonstration; the algorithm does not discover the number of clusters itself.

`labels` stores each sample's assigned cluster, and `centroids` stores the cluster centers in scaled feature space. Cluster numbers are arbitrary identifiers, not species labels.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X_scaled)

labels = kmeans.labels_
centroids = kmeans.cluster_centers_


### Visualize the clusters

Colors show cluster assignments and red crosses show the centroids. The model uses all four features, while this plot shows only the first two standardized features. Clusters that overlap in this view may be better separated in the other dimensions. Axis values are standardized values, not the original measurements.

In [ ]:
# Using first two features for 2D plot
plt.figure(figsize=(8,6))
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=labels, cmap='viridis', s=50)
plt.scatter(centroids[:,0], centroids[:,1], color='red', marker='X', s=200, label='Centroids')
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title('K-Means Clustering on Iris Dataset')
plt.legend()
plt.show()


### Evaluate separation with the silhouette score

The silhouette score compares how close each sample is to its own cluster with how close it is to the nearest other cluster. Scores range from -1 to 1: values near 1 suggest well-separated clusters, values near 0 indicate overlap, and negative values suggest samples may fit better in another cluster.

This score uses the scaled features and cluster assignments, not the known species labels. It measures separation rather than species-classification accuracy.

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_scaled, labels)
print("Silhouette Score:", score)


### Assign a new flower to a cluster

Provide the four measurements in the same order as the training features. Reuse the fitted scaler and K-means model to assign the new sample to the nearest centroid. The returned number identifies a cluster; it does not directly name a flower species.

In [ ]:
# Example new iris sample
new_sample = np.array([[5.0, 3.5, 1.5, 0.3]])  # features
new_sample_scaled = scaler.transform(new_sample)
predicted_cluster = kmeans.predict(new_sample_scaled)
print("New sample assigned to cluster:", predicted_cluster[0])
